In [6]:
import polars as pl
import json

In [10]:
with open('./data/result.json', 'r', encoding='utf-8') as f:
    tg_json = json.load(f)

data = []

for message in tg_json['messages']:
    msg_type = message.get('type')
    date = message.get('date')
    
    sender = message.get('from') or message.get('actor') or message.get('actor_id') or message.get('from_id', 'Unknown')
    if sender in (YOUR_ID, YOUR_USERNAME):
        sender = 'gpt'
    else:
        sender = 'human'
    
    raw_text = message.get('text', '')
    if isinstance(raw_text, str):
        text = raw_text
    elif isinstance(raw_text, list):
        parts = []
        for part in raw_text:
            if isinstance(part, dict):
                parts.append(part.get('text', ''))
            else:
                parts.append(str(part))
        text = ''.join(parts)
    else:
        text = ''
    
    data.append({
        "msg_type": msg_type,
        "date": date,
        "role": sender,
        "text": text
    })
    
df = pl.DataFrame(data)
df = df.with_columns(pl.col('date').str.to_datetime())
df = df.filter(pl.col('text') != '', pl.col('msg_type') == 'message')

df = df.with_columns(pl.col('date').diff().alias('time_delta'))

df = df.with_columns(
    pl.when(
        (pl.col('role') != pl.col('role').shift()) | (pl.col('time_delta').dt.total_seconds() > 300)
    )
    .then(True)
    .otherwise(False)
    .alias('break')
)

df = df.with_columns(pl.col('break').cast(pl.Int64).cum_sum().alias('group_id'))

In [39]:
grouped_df = df.group_by(pl.col('group_id')).agg(
    pl.col('role').first(),
    pl.col('text').sort_by('date').str.join('\n\n'),
    pl.col('date').sort_by('date').first()
).sort('date')

grouped_df = grouped_df\
    .with_columns(pl.col('role').rle_id().alias('chat_block_id'))\
    .group_by('chat_block_id')\
    .agg(
        pl.col('role').first(),
        pl.col('text').sort_by('date').str.join('\n\n'),
        pl.col('date').sort_by('date').first()
    ).sort('date')

In [ ]:
grouped_dicts = grouped_df.to_dicts()

dataset = []

for i in range(len(grouped_dicts)):
    if grouped_dicts[i]['role'] == 'human':
        continue
    else:
        start_index = max(0, i - WINDOW)
        win_slice = grouped_dicts[start_index:i+1]
        
        if win_slice[0]['role'] == 'gpt' and len(win_slice) > 1:
            win_slice = win_slice[1:]
        
        curr_conv = [{"from": "system", "value": SYSTEM_PROMPT}]
        for msg in win_slice:
            curr_conv.append({"from": msg['role'], "value": msg['text']})
        dataset.append({"conversations": curr_conv})

with open('./data/dataset.jsonl', 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')